In [16]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [17]:
from langchain_groq import ChatGroq

chatllm=ChatGroq(model="openai/gpt-oss-120b")


In [18]:
chatllm.invoke("hi buddy").content

'Hey there! How’s it going? 😊'

In [19]:
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import BaseMessage
from langchain_core.messages import AnyMessage

class Graphstate(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [20]:
from langchain_core.messages import AnyMessage,HumanMessage,AIMessage

In [21]:
def llm_call(state: Graphstate) -> dict:
    response = chatllm.invoke(state["messages"])
    
    return {
        "messages": state["messages"] + [response]
    }

token counter


In [22]:
def token_counter(state: Graphstate) -> dict:
    last_msg = state["messages"][-1]
    text = last_msg.content

    token_number = len(text.split())
    print("Token count:", token_number)

    return {"messages": state["messages"]}

sequence using stategraph

In [23]:
from langgraph.graph import StateGraph

In [24]:
function=StateGraph(Graphstate)

In [25]:
function.add_node("llm_call",llm_call)
function.add_node("token_counter",token_counter)

In [26]:
function.set_entry_point("llm_call")
function.add_edge("llm_call","token_counter")
function.set_finish_point("token_counter")

In [27]:
add=function.compile()

In [28]:
result = add.invoke({
    "messages": [HumanMessage(content="hi, this is aryan")]
})

Token count: 9


In [29]:
result

{'messages': [HumanMessage(content='hi, this is aryan', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='hi, this is aryan', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello Aryan! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "hi, this is aryan". Probably they are greeting and introducing themselves. We should respond politely, ask how we can help. No disallowed content. Just a friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 77, 'total_tokens': 140, 'completion_time': 0.133214909, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.025989865, 'prompt_tokens_details': None, 'queue_time': 0.161067994, 'total_time': 0.159204774}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_84efe82cfb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cff70-c167-74f1-8ee

In [30]:
for me in result["messages"]:
    print(type(me).__name__,":",me.content)

HumanMessage : hi, this is aryan
HumanMessage : hi, this is aryan
AIMessage : Hello Aryan! 👋 How can I assist you today?
HumanMessage : hi, this is aryan
HumanMessage : hi, this is aryan
AIMessage : Hello Aryan! 👋 How can I assist you today?


tool calling

In [31]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [32]:
api_wrapers=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)

In [33]:
wiki_tool=WikipediaQueryRun(api_wrapper=api_wrapers)

In [34]:
wiki_tool.run({"query":"agents in ai"})

'Page: AI agent\nSummary: In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a class of intelligent agents distinguished by their ability to operate autonomously in complex environments. Agentic AI tools prioritize decision-making over content creation and do not require continuous oversight.\n\n\n\nPage: Manus (AI agent)\nSummary: Manus (meaning hand in Latin) is an autonomous artificial intelligence agent developed by Butterfly '

In [35]:
import os 
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
from langchain_community.tools.tavily_search import TavilySearchResults

In [36]:
TAVILY_API_KEY

'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'

In [39]:
tools=TavilySearchResults(tavily_api_key=TAVILY_API_KEY)

In [41]:
tools.invoke({"query":""})
tools.invoke({"query":"what is gen ai"})

[{'title': 'What is Generative AI? - San Francisco State University',
  'url': 'https://ai.sfsu.edu/what-is-gen-ai',
  'content': 'SAN FRANCISCO STATE UNIVERSITY |   AI\n\n# What is Generative AI?\n\nGenerative Artificial Intelligence (Gen AI) refers to a type of artificial intelligence that can create new content, such as text, images, audio or video, by learning patterns from large sets of sample material. Examples of tools that use this technology include chatbots such as ChatGPT, Microsoft Copilot (formerly known as Bing Chat) and Google Gemini (formerly known as Bard), and image generators like Midjourney and DALL-E. [...] While traditional AI has been used to analyze data, make predictions and perform specific tasks using predefined rules, Gen AI goes beyond these capabilities by creating new patterns from existing ones. It’s the difference between an AI that can play chess based on established rules and an AI that can generate concepts for entirely new board games.\n\nDALL-E 3 c

In [ ]:
from langchain_community.tools import YouTubeSearchTool

In [ ]:
newtools=YouTubeSearchTool()

In [49]:
tools.run("apna college")

[{'title': 'Apna College',
  'url': 'https://in.linkedin.com/company/apna-college',
  'content': "# Apna College\nNo.1 for online Tech Placements & Internships Thousands of students Placed 🚀 7 Million+ Coders on YouTube\nE-Learning Providers • Delhi • 172,946 followers • 51-200 employees\n\n## Overview\nNo.1 for Online Tech Placements Preparation Thousands of students placed successfully. Save time & study only what's needed for placements.\n\n### Website\n\n### Crunchbase\nN/A\n### LinkedIn\n\n### Industry\nE-Learning Providers\n\n### Company Size\n51-200 employees  \n1514 associated members (LinkedIn members who've listed Apna College as their current workplace on their profile)\n\n### Founded\n2020\n\n### Funding\nLast Round Date: N/A\nLast Round Type: N/A\nTotal Rounds: N/A\nLast Round Raised: N/A\n\n### Investors\nN/A\n\n### Specialties\nN/A\n\n## Locations\nDelhi, Delhi, IN\n\n### Get Directions\nDirections [...] College) is extremely helpful for practice — every question is impo